# Analytics Project Group Alpha
## Exploratory Data Analysis: Dynamic Pricing & Purchase Prediction
**CRISP-DM Phase 2 | Data Understanding*  
**Lecturer: Prof. Andreas Reber | Source: DMC 2017 (adapted)**

---
### Notebook Structure
1. [Setup & Data Loading](#1-setup)
2. [Initial Data Inspection](#2-inspection)
3. [Descriptive Statistics](#3-stats)
4. [Missing Value Analysis](#4-missing)
5. [Target Variable Analysis (Class Distribution)](#5-target)
6. [Feature Distributions](#6-features)
7. [Relationship Analysis (Features vs. Target)](#7-relationships)
8. [Correlation Analysis](#8-correlation)
9. [Data Quality Summary](#9-quality)

> **Note:** This notebook assumes `train.csv` and `items.csv` are located in `./data/`.  
> Download both files from Moodle and place them there before running.

---
## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, Markdown
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Global Plot Style ────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.titlesize': 14,
})

SEED = 42
np.random.seed(SEED)

print('Libraries loaded successfully.')

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────
# Adjust paths if needed
TRAIN_PATH = './data/train.csv'
ITEMS_PATH = './data/items.csv'

print('Loading train.csv (~2.5M rows — this may take a moment)...')
train = pd.read_csv(TRAIN_PATH, sep='|')

print('Loading items.csv...')
items = pd.read_csv(ITEMS_PATH, sep='|')

# ── Sanity check: confirm columns parsed correctly ──────────────────────────
print(f'\ntrain.csv columns ({len(train.columns)}): {list(train.columns)}')
print(f'items.csv columns ({len(items.columns)}): {list(items.columns)}')

# Auto-detect if still single column (wrong separator fallback)
if len(train.columns) == 1:
    print('\n⚠️  Still one column — trying comma separator as fallback...')
    train = pd.read_csv(TRAIN_PATH, sep=',')
    items = pd.read_csv(ITEMS_PATH, sep=',')
    print(f'train.csv columns: {list(train.columns)}')
    print(f'items.csv columns: {list(items.columns)}')

print(f'\ntrain.csv  : {train.shape[0]:,} rows × {train.shape[1]} columns')
print(f'items.csv  : {items.shape[0]:,} rows × {items.shape[1]} columns')

---
## 2. Initial Data Inspection <a id='2-inspection'></a>
### 2.1 Schema & Variable Dictionary

In [ ]:
# ── Variable Dictionary (from project brief) ─────────────────────────────────
train_dict = {
    'lineID':           'Unique identifier for each user action (key)',
    'day':              'Day in the observed period',
    'pid':              'Product number (join key to items.csv)',
    'adFlag':           'Advertising flag: product in marketing campaign {0,1}',
    'availability':     'Product availability status {1,2,3,4}',
    'competitorPrice':  'Lowest competitor price (positive decimal, may be missing)',
    'click':            'Click flag {0,1}',
    'basket':           'Basket flag {0,1}',
    'order':            '>>> TARGET: Order flag {0,1} <<<',
    'price':            'Product price (positive decimal)',
    'revenue':          'Revenue (only non-zero when order=1 — LEAKAGE RISK)',
}

items_dict = {
    'pid':              'Product number (join key)',
    'manufacturer':     'Manufacturer ID (anonymized)',
    'group':            'Product group (string)',
    'content':          'Package content (number or NxN format)',
    'unit':             'Unit of measurement',
    'pharmForm':        'Dosage form (3-char string)',
    'genericProduct':   'Generic drug flag {0,1}',
    'salesIndex':       'Dispensing regulation code',
    'category':         'Main shop category (integer)',
    'campainIndex':     'Campaign label {A, B, C}',
    'rrp':              'Reference retail price (positive decimal)',
}

print('=== train.csv Variable Dictionary ===')
for k, v in train_dict.items():
    print(f'  {k:<20} {v}')

print('\n=== items.csv Variable Dictionary ===')
for k, v in items_dict.items():
    print(f'  {k:<20} {v}')

In [ ]:
# ── First look at raw data ────────────────────────────────────────────────────
print('--- train.csv: first 5 rows ---')
display(train.head())

print('\n--- items.csv: first 5 rows ---')
display(items.head())

In [ ]:
# ── Data Types ────────────────────────────────────────────────────────────────
print('--- train.csv: dtypes & memory ---')
display(train.dtypes.rename('dtype').to_frame())

print(f'\nMemory usage: {train.memory_usage(deep=True).sum() / 1e6:.1f} MB')

print('\n--- items.csv: dtypes ---')
display(items.dtypes.rename('dtype').to_frame())

In [ ]:
# ── Check for duplicate rows ──────────────────────────────────────────────────
train_dups  = train.duplicated().sum()
items_dups  = items.duplicated().sum()
lineid_dups = train['lineID'].duplicated().sum()

print(f'train.csv  — duplicate rows    : {train_dups:,}')
print(f'train.csv  — duplicate lineIDs : {lineid_dups:,}')
print(f'items.csv  — duplicate rows    : {items_dups:,}')
print(f'items.csv  — unique PIDs       : {items["pid"].nunique():,}')

---
## 3. Descriptive Statistics <a id='3-stats'></a>

In [ ]:
# ── Numerical summary ─────────────────────────────────────────────────────────
print('=== train.csv: Descriptive Statistics (numerical) ===')
display(train.describe(percentiles=[.05, .25, .50, .75, .95]).round(4))

In [ ]:
print('=== items.csv: Descriptive Statistics ===')
display(items.describe(include='all').round(4))

In [ ]:
# ── Temporal range ────────────────────────────────────────────────────────────
print(f'Day range in train.csv  : {train["day"].min()} → {train["day"].max()} ({train["day"].nunique()} unique days)')
print(f'Unique products (PIDs)  : {train["pid"].nunique():,}')
print(f'Price range             : {train["price"].min():.2f} – {train["price"].max():.2f}')
print(f'Revenue range           : {train["revenue"].min():.2f} – {train["revenue"].max():.2f}')
print(f'Competitor price range  : {train["competitorPrice"].min():.2f} – {train["competitorPrice"].max():.2f}')

---
## 4. Missing Value Analysis <a id='4-missing'></a>

In [ ]:
def missing_report(df, name):
    """Returns a DataFrame summarising missing values."""
    total   = df.isnull().sum()
    pct     = (total / len(df) * 100).round(2)
    dtype   = df.dtypes
    report  = pd.DataFrame({'Missing (n)': total, 'Missing (%)': pct, 'dtype': dtype})
    report  = report[report['Missing (n)'] > 0].sort_values('Missing (%)', ascending=False)
    print(f'=== Missing Values: {name} ===')
    if report.empty:
        print('  ✅ No missing values found.')
    else:
        display(report)
    return report

train_missing = missing_report(train, 'train.csv')
items_missing = missing_report(items, 'items.csv')

In [ ]:
# ── Visual: Missing Value Heatmap ────────────────────────────────────────────
cols_with_missing = train.columns[train.isnull().any()].tolist()

if cols_with_missing:
    # Sample for visualisation performance
    sample_size = min(5000, len(train))
    sample = train[cols_with_missing].sample(sample_size, random_state=SEED)

    fig, ax = plt.subplots(figsize=(max(8, len(cols_with_missing) * 1.5), 5))
    sns.heatmap(sample.isnull(), cbar=False, yticklabels=False,
                cmap='viridis', ax=ax)
    ax.set_title(f'Missing Value Pattern — train.csv (n={sample_size:,} sample)')
    ax.set_xlabel('Column')
    plt.tight_layout()
    Path('./figures').mkdir(parents=True, exist_ok=True)
    plt.savefig('./figures/fig01_missing_heatmap.png', bbox_inches='tight')
    plt.show()
else:
    print('No missing values to plot for train.csv.')

In [ ]:
# ── Missing values: competitorPrice by availability status ──────────────────
if 'competitorPrice' in train.columns:
    cp_miss_by_avail = (
        train.groupby('availability')['competitorPrice']
        .apply(lambda x: x.isnull().mean() * 100)
        .rename('CompetitorPrice Missing (%)')
        .reset_index()
    )
    print('competitorPrice missingness by availability status:')
    display(cp_miss_by_avail)

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=cp_miss_by_avail, x='availability', y='CompetitorPrice Missing (%)',
                palette='Blues_d', ax=ax)
    ax.set_title('% Missing competitorPrice by Availability Status')
    ax.set_ylabel('Missing (%)')
    plt.tight_layout()
    plt.savefig('./figures/fig02_missing_competitor_by_avail.png', bbox_inches='tight')
    plt.show()

---
## 5. Target Variable Analysis <a id='5-target'></a>
### 5.1 Class Distribution: order / basket / click

In [ ]:
# ── Counts & proportions ──────────────────────────────────────────────────────
for col in ['click', 'basket', 'order']:
    counts = train[col].value_counts().sort_index()
    pct    = (counts / len(train) * 100).round(3)
    print(f'{col:>10}  →  0: {counts.get(0, 0):>10,} ({100 - pct.get(1,0):.3f}%)  '
          f'|  1: {counts.get(1, 0):>8,} ({pct.get(1,0):.3f}%)')

In [ ]:
# ── Visual: Class imbalance ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colours = ['#4C72B0', '#DD8452']

for ax, col in zip(axes, ['click', 'basket', 'order']):
    counts = train[col].value_counts().sort_index()
    bars = ax.bar(counts.index.astype(str), counts.values, color=colours, edgecolor='white', linewidth=0.8)
    ax.set_title(f'Distribution of "{col}"')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    for bar in bars:
        pct = bar.get_height() / len(train) * 100
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f'{pct:.2f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Class Distribution: User Actions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig03_class_distribution.png', bbox_inches='tight')
plt.show()

# ── Imbalance ratio ──────────────────────────────────────────────────────────
order_counts = train['order'].value_counts()
ratio = order_counts[0] / order_counts.get(1, 1)
print(f'\nClass imbalance ratio (0:1) for "order": {ratio:.1f} : 1')
print('⚠️  Significant imbalance — must apply SMOTE or class_weight="balanced" in modelling phase.')

In [ ]:
# ── Mutual exclusivity check ──────────────────────────────────────────────────
# Per brief: only one of click/basket/order can be 1 per row
action_sum = train[['click', 'basket', 'order']].sum(axis=1)
violations = (action_sum > 1).sum()
print(f'Rows where more than one action flag = 1 (should be 0): {violations:,}')
assert violations == 0, '⚠️ Mutual exclusivity violated — review data!'
print('✅ Mutual exclusivity confirmed: exactly one action flag per row.')

---
## 6. Feature Distributions <a id='6-features'></a>
### 6.1 Numerical Features

In [ ]:
# ── Price & CompetitorPrice distribution ────────────────────────────────────
num_cols = ['price', 'competitorPrice', 'revenue']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, num_cols):
    data = train[col].dropna()
    # Cap at 99th percentile for visual clarity
    upper = data.quantile(0.99)
    data  = data[data <= upper]
    ax.hist(data, bins=60, color='#4C72B0', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.set_title(f'Distribution of "{col}" (capped at 99th pct)')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig04_numerical_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Price vs Competitor Price scatter ────────────────────────────────────────
sample = train[['price', 'competitorPrice', 'order']].dropna().sample(
    min(5000, len(train)), random_state=SEED
)

fig, ax = plt.subplots(figsize=(8, 6))
colours_map = {0: '#AEC6E8', 1: '#E8503A'}
for order_val, group in sample.groupby('order'):
    ax.scatter(group['competitorPrice'], group['price'],
               c=colours_map[order_val], alpha=0.4, s=8,
               label=f'order={order_val}')

# Parity line (price == competitor price)
lim = max(sample['price'].quantile(0.99), sample['competitorPrice'].quantile(0.99))
ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2, label='price = competitorPrice')

ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel('Competitor Price')
ax.set_ylabel('Product Price')
ax.set_title('Price vs. Competitor Price (sample n=5,000)')
ax.legend()
plt.tight_layout()
plt.savefig('./figures/fig05_price_vs_competitor.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Price-to-RRP ratio (after merging items) ─────────────────────────────────
campaign_col = 'campainIndex' if 'campainIndex' in items.columns else (
    'campaignIndex' if 'campaignIndex' in items.columns else None
)

item_cols = ['pid', 'rrp', 'genericProduct', 'category', 'group']
if campaign_col is not None:
    item_cols.append(campaign_col)

merged = train.merge(items[item_cols], on='pid', how='left')

# Keep downstream notebook compatibility (uses "campainIndex")
if campaign_col == 'campaignIndex':
    merged = merged.rename(columns={'campaignIndex': 'campainIndex'})

merged['price_ratio'] = merged['price'] / merged['rrp'].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(9, 5))
data = merged['price_ratio'].dropna()
data = data[data.between(0.01, 5)]  # remove extreme outliers for display
ax.hist(data, bins=80, color='#55A868', edgecolor='white', linewidth=0.4, alpha=0.85)
ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='price = rrp')
ax.set_title('Distribution of Price-to-RRP Ratio (price / rrp)')
ax.set_xlabel('price / rrp')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('./figures/fig06_price_ratio.png', bbox_inches='tight')
plt.show()

pct_below_rrp = (merged['price_ratio'] < 1).mean() * 100
print(f'{pct_below_rrp:.1f}% of products are priced below their RRP.')

### 6.2 Categorical Features

In [ ]:
# ── Availability & adFlag distributions ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Availability
avail_counts = train['availability'].value_counts().sort_index()
axes[0].bar(avail_counts.index.astype(str), avail_counts.values,
            color='#4C72B0', edgecolor='white')
axes[0].set_title('Availability Status Distribution')
axes[0].set_xlabel('Availability Code')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(avail_counts.values):
    axes[0].text(i, v * 1.01, f'{v/len(train)*100:.1f}%', ha='center', fontsize=9)

# adFlag
adflag_counts = train['adFlag'].value_counts().sort_index()
axes[1].bar(adflag_counts.index.astype(str), adflag_counts.values,
            color=['#AEC6E8', '#DD8452'], edgecolor='white')
axes[1].set_title('Advertising Flag (adFlag) Distribution')
axes[1].set_xlabel('adFlag (0=not advertised, 1=advertised)')
axes[1].set_ylabel('Count')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(adflag_counts.values):
    axes[1].text(i, v * 1.01, f'{v/len(train)*100:.1f}%', ha='center', fontsize=9)

plt.suptitle('Categorical Feature Distributions (train.csv)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig07_categorical_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Campaign Index & Generic product flag (items.csv) ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# campainIndex / campaignIndex (handle both spellings safely)
campaign_col = 'campainIndex' if 'campainIndex' in items.columns else (
    'campaignIndex' if 'campaignIndex' in items.columns else None
)

if campaign_col is not None:
    camp_counts = items[campaign_col].fillna('Missing').value_counts().sort_index()
    axes[0].bar(camp_counts.index.astype(str), camp_counts.values,
                color=['#4C72B0', '#DD8452', '#55A868', '#999999'][:len(camp_counts)],
                edgecolor='white')
    axes[0].set_title('Campaign Index (items.csv)')
    axes[0].set_xlabel(campaign_col)
    axes[0].set_ylabel('Count')
else:
    axes[0].text(0.5, 0.5, 'No campaign column found', ha='center', va='center')
    axes[0].set_axis_off()

# genericProduct
gen_counts = items['genericProduct'].value_counts().sort_index()
axes[1].bar(gen_counts.index.astype(str), gen_counts.values,
            color=['#AEC6E8', '#E8503A'], edgecolor='white')
axes[1].set_title('Generic Product Flag (items.csv)')
axes[1].set_xlabel('genericProduct (0=branded, 1=generic)')

# Top 15 product groups
top_groups = items['group'].value_counts().head(15)
axes[2].barh(top_groups.index[::-1], top_groups.values[::-1], color='#4C72B0')
axes[2].set_title('Top 15 Product Groups (items.csv)')
axes[2].set_xlabel('Count')

plt.suptitle('Product Attribute Distributions (items.csv)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig08_items_distributions.png', bbox_inches='tight')
plt.show()

### 6.3 Temporal Analysis

In [ ]:
# ── Orders per day ────────────────────────────────────────────────────────────
daily_orders = train.groupby('day')['order'].agg(['sum', 'mean']).reset_index()
daily_orders.columns = ['day', 'order_count', 'order_rate']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(daily_orders['day'], daily_orders['order_count'],
             color='#4C72B0', linewidth=1.2)
axes[0].fill_between(daily_orders['day'], daily_orders['order_count'],
                     alpha=0.15, color='#4C72B0')
axes[0].set_title('Number of Orders per Day')
axes[0].set_ylabel('Order Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].plot(daily_orders['day'], daily_orders['order_rate'] * 100,
             color='#DD8452', linewidth=1.2)
axes[1].fill_between(daily_orders['day'], daily_orders['order_rate'] * 100,
                     alpha=0.15, color='#DD8452')
axes[1].set_title('Order Rate (%) per Day')
axes[1].set_ylabel('Order Rate (%)')
axes[1].set_xlabel('Day')

plt.suptitle('Temporal Order Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig09_temporal_orders.png', bbox_inches='tight')
plt.show()

---
## 7. Relationship Analysis: Features vs. Target <a id='7-relationships'></a>

In [ ]:
# ── Order rate by adFlag ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['adFlag', 'availability', 'genericProduct']):
    if col == 'genericProduct':
        data_col = merged[col]
        order_col = merged['order']
    else:
        data_col = train[col]
        order_col = train['order']

    order_rate = pd.DataFrame({'col': data_col, 'order': order_col})\
                   .groupby('col')['order'].mean() * 100
    order_rate = order_rate.sort_index()

    bars = ax.bar(order_rate.index.astype(str), order_rate.values,
                  color='#4C72B0', edgecolor='white')
    ax.set_title(f'Order Rate by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Order Rate (%)')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f'{bar.get_height():.2f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Order Rate by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig10_order_rate_categorical.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Order rate by campainIndex (from items.csv) ───────────────────────────────
camp_order = merged.groupby('campainIndex')['order'].mean() * 100
camp_order = camp_order.sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(camp_order.index.astype(str), camp_order.values,
              color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white')
ax.set_title('Order Rate by Campaign Index')
ax.set_xlabel('campainIndex')
ax.set_ylabel('Order Rate (%)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
            f'{bar.get_height():.3f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('./figures/fig11_order_rate_campaign.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Price distribution: ordered vs not ordered ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['price', 'competitorPrice']):
    for label, group in train.groupby('order'):
        data = group[col].dropna()
        cap  = data.quantile(0.99)
        data = data[data <= cap]
        ax.hist(data, bins=60, alpha=0.55,
                label=f'order={label}',
                density=True,
                color='#DD8452' if label == 1 else '#4C72B0')
    ax.set_title(f'{col} — Ordered vs. Not Ordered')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Price Distribution by Order Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig12_price_by_order.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Box plots: price & price_ratio by order ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Price boxplot (capped at 99th pct)
cap = train['price'].quantile(0.99)
plot_data = train[train['price'] <= cap]
sns.boxplot(data=plot_data, x='order', y='price', palette=['#4C72B0', '#DD8452'],
            ax=axes[0], width=0.5)
axes[0].set_title('Price by Order Status')
axes[0].set_xlabel('order')

# Price ratio boxplot
merged['price_ratio_cap'] = merged['price_ratio'].clip(0, 3)
sns.boxplot(data=merged.dropna(subset=['price_ratio_cap']),
            x='order', y='price_ratio_cap',
            palette=['#4C72B0', '#DD8452'], ax=axes[1], width=0.5)
axes[1].set_title('Price/RRP Ratio by Order Status')
axes[1].set_xlabel('order')
axes[1].set_ylabel('price / rrp (capped at 3)')

plt.suptitle('Pricing Features vs. Order Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig13_price_boxplots.png', bbox_inches='tight')
plt.show()

---
## 8. Correlation Analysis <a id='8-correlation'></a>

In [ ]:
# ── Prepare merged numeric dataset ────────────────────────────────────────────
# Encode categoricals for correlation
merged_enc = merged.copy()
for col in ['campainIndex']:
    if col in merged_enc.columns:
        merged_enc[col] = pd.Categorical(merged_enc[col]).codes

numeric_cols = [
    'day', 'adFlag', 'availability', 'competitorPrice', 'click', 'basket', 'order',
    'price', 'rrp', 'price_ratio', 'genericProduct', 'salesIndex', 'category',
    'campainIndex'
]
numeric_cols = [c for c in numeric_cols if c in merged_enc.columns]

corr_matrix = merged_enc[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation Matrix — Merged Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/fig14_correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Top correlations with target variable 'order' ────────────────────────────
if 'order' in corr_matrix.columns:
    order_corr = (
        corr_matrix['order']
        .drop('order')
        .drop(['click', 'basket', 'revenue'], errors='ignore')  # direct action cols & leakage
        .abs()
        .sort_values(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    colours = ['#4C72B0' if v >= 0 else '#DD8452'
               for v in corr_matrix['order'].drop(['order', 'click', 'basket'],
                                                   errors='ignore')[order_corr.index]]
    ax.barh(order_corr.index[::-1], order_corr.values[::-1], color=colours[::-1])
    ax.set_title('Feature Correlation with Target Variable (|r| with order)')
    ax.set_xlabel('|Pearson r|')
    ax.axvline(0.05, color='red', linestyle='--', linewidth=1, label='r=0.05 threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('./figures/fig15_target_correlations.png', bbox_inches='tight')
    plt.show()

    print('Top 10 features correlated with order:')
    display(order_corr.head(10).round(4).to_frame('|r| with order'))

In [ ]:
# ── Scatter matrix (selected features) ───────────────────────────────────────
scatter_cols = ['price', 'competitorPrice', 'price_ratio', 'rrp', 'order']
scatter_cols = [c for c in scatter_cols if c in merged.columns]

sample_scatter = merged[scatter_cols].dropna().sample(
    min(3000, len(merged)), random_state=SEED
)

fig = sns.pairplot(
    sample_scatter, hue='order', diag_kind='kde',
    plot_kws={'alpha': 0.3, 's': 8},
    palette={0: '#4C72B0', 1: '#DD8452'}
)
fig.fig.suptitle('Scatter Matrix — Pricing Features by Order Status',
                 y=1.02, fontsize=13, fontweight='bold')
plt.savefig('./figures/fig16_scatter_matrix.png', bbox_inches='tight')
plt.show()

---
## 9. Data Quality Summary <a id='9-quality'></a>

In [ ]:
# ── Outlier Detection: IQR method ────────────────────────────────────────────
def iqr_outliers(df, col):
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((df[col] < lower) | (df[col] > upper)).sum()
    return n_out, lower, upper

print('Outlier Analysis (IQR method):')
print(f'{"Column":<20} {"N Outliers":>12} {"% of Rows":>10} {"Lower bound":>14} {"Upper bound":>14}')
print('-' * 76)
for col in ['price', 'competitorPrice', 'rrp']:
    if col in train.columns or col in merged.columns:
        src = merged if col == 'rrp' else train
        if col not in src.columns:
            continue
        data   = src[col].dropna()
        n, lo, hi = iqr_outliers(pd.DataFrame({col: data}), col)
        print(f'{col:<20} {n:>12,} {n/len(data)*100:>9.2f}% {lo:>14.2f} {hi:>14.2f}')

In [ ]:
# ── Check: revenue = 0 when order = 0 ────────────────────────────────────────
revenue_leak = train[(train['order'] == 0) & (train['revenue'] > 0)]
print(f'Rows where order=0 but revenue>0 (data leakage check): {len(revenue_leak):,}')
if len(revenue_leak) == 0:
    print('✅ Revenue is strictly 0 when order=0 — confirmed leakage source, must be dropped.')
else:
    print('⚠️  Revenue has non-zero values even for non-orders — investigate before modelling.')

In [ ]:
# ── Content column check (items.csv) ─────────────────────────────────────────
if 'content' in items.columns:
    # Check for mixed types (e.g., "5X10" vs numeric)
    content_numeric  = pd.to_numeric(items['content'], errors='coerce')
    n_numeric        = content_numeric.notna().sum()
    n_string_format  = items['content'].astype(str).str.contains(r'\d+X\d+', na=False).sum()
    n_other          = len(items) - n_numeric - n_string_format

    print('Content column analysis:')
    print(f'  Numeric values      : {n_numeric:,}')
    print(f'  "NxN" format values : {n_string_format:,}')
    print(f'  Other / missing     : {n_other:,}')
    print('  → Requires parsing in Data Preparation phase.')

In [ ]:
# ── PID join completeness ─────────────────────────────────────────────────────
pids_in_train = set(train['pid'].unique())
pids_in_items = set(items['pid'].unique())

in_both   = len(pids_in_train & pids_in_items)
only_train = len(pids_in_train - pids_in_items)
only_items = len(pids_in_items - pids_in_train)

print('PID Join Coverage:')
print(f'  PIDs in both train & items : {in_both:,}')
print(f'  PIDs only in train         : {only_train:,}  ← will have NaN for item attributes')
print(f'  PIDs only in items         : {only_items:,}  ← never purchased in training period')

In [ ]:
# ── Final Data Quality Report ─────────────────────────────────────────────────
print('=' * 70)
print('  DATA QUALITY SUMMARY — CRISP-DM Phase 2 Findings')
print('=' * 70)

quality_issues = [
    ('Missing values', 'competitorPrice contains missing values. Strategy: median imputation per product group + binary indicator flag.'),
    ('Data leakage',   '"revenue" is 0 when order=0 and non-zero only when order=1 → MUST be dropped before modelling.'),
    ('Class imbalance', f'Order rate is very low (< 5% expected). Strategy: SMOTE or class_weight="balanced".'),
    ('Mixed types',    '"content" in items.csv contains both numeric and "NxN" string formats → requires parsing.'),
    ('Scale levels',   'Mix of nominal (group, pharmForm), ordinal (availability, salesIndex), and proportional (price, rrp) features.'),
    ('Join coverage',  f'{only_train} PIDs in train.csv have no match in items.csv → careful handling needed.'),
]

for i, (issue, desc) in enumerate(quality_issues, 1):
    print(f'  [{i}] {issue}')
    print(f'       → {desc}')
    print()

print('=' * 70)
print('  All issues documented. See Data Preparation phase for resolution.')
print('=' * 70)

In [ ]:
# ── Save merged dataset for use in Data Preparation notebook ─────────────────
import os
os.makedirs('./data/processed', exist_ok=True)
os.makedirs('./figures', exist_ok=True)

merged.to_csv('./data/processed/merged_raw.csv', index=False)
print('Merged dataset saved to: ./data/processed/merged_raw.csv')
print(f'Shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns')
print()
print('EDA complete. All figures saved to ./figures/')
print('Proceed to: DataPreparation.ipynb')